# Projeto 01 — Agente Inteligente de Otimização de Rotas de Distribuição Alimentar Local
## Unidade Curricular de Inteligência Artificial — IPCA (2026/2027)
### Resolução de Problemas de Pickup-and-Delivery com Janelas Temporais e Capacidade (PDPTW)

---

### Grupo 04 — `IA26_P01_G04`
* **Gonçalo Correia** — Nº 22840
* **Bruno Dantas** — Nº 27958
* **Ricardo Ferreira** — Nº 25426

**Repositório Oficial:** [https://github.com/BrunoDantas1/IA26_P01_G04.git](https://github.com/BrunoDantas1/IA26_P01_G04.git)

---

## 1. Introduction

### 1.1. Contexto do Problema e Enquadramento
A transição para cadeias de abastecimento alimentar curtas e sustentáveis coloca desafios logísticos complexos. Pequenos produtores agrícolas locais operam com margens temporais reduzidas de frescura dos produtos, enquanto os consumidores finais e mercados municipais exigem entregas em horários de conveniência bem definidos.

Neste projeto desenvolvemos um **Agente Inteligente de Planeamento de Rotas** para distribuição alimentar local na região do distrito de **Braga** e concelho da **Póvoa de Varzim**. O problema enquadra-se formalmente na classe dos **Pickup-and-Delivery Problems with Time Windows and Capacity Constraints (PDPTW)**, que combinam restrições espaciais, de precedência, temporais e de capacidade.

### 1.2. Atores e Componentes do Sistema
1. **Produtores Locais (*Producers*)**: Locais de origem onde os produtos agrícolas são colhidos/embalados e disponibilizados dentro da janela regulamentar $[\text{availability\_start}, \text{availability\_end}]$.
2. **Centros de Recolha / Armazéns (*CollectionCenters*)**: Plataformas logísticas centrais onde os bens podem ser agregados e que servem de depósito base permanente da frota de veículos.
3. **Locais de Entrega (*DeliveryLocations / Customers*)**: Clientes residenciais ou mercados municipais com janelas de entrega $[\text{delivery\_start}, \text{delivery\_end}]$.
4. **Frota de Veículos (*Vehicles*)**: Carrinhas comerciais com capacidades máximas específicas (1444.7 kg e 1009.8 kg) que realizam as viagens.
5. **Matriz de Distâncias e Tempos (*DistanceMatrix*)**: Baseada na rede de paragens unificadas de `locations.csv`, fornecendo distâncias em km (distância ortodrómica corrigida com fator rodoviário de 1.3) e tempos de viagem em minutos calculados a uma velocidade de 45 km/h.

### 1.3. Objetivos do Projeto e Divisão de Tarefas
* **Secção 1 — Introdução**: Contextualização, identificação e definição do sistema logístico (preenchida).
* **Secção 2 — Exploratory Data Analysis (EDA)**: Carregamento do dataset, validação de integridade referencial e análise da procura face à frota *(incluída nesta entrega)*.
* **Secção 3 — Modelação e Testes de Viabilidade**: Definição das classes `Request`, `Vehicle`, `State`, `Problem`, função de custo e validador estrito de rotas com testes manuais (*hand-written routes*) *(incluída nesta entrega)*.
* **Secções 4 e 5 — Problem Solving & Conclusão**: Algoritmos de pesquisa informada (Greedy, A*), pesquisa local (Hill-Climbing) e CSP com coordenação de frota *(a desenvolver na Entrega Final)*.


## 2. Exploratory Data Analysis (EDA)

Os valores seguintes foram confirmados nos 11 CSV de `data`. A análise completa consta de `EDA.ipynb` e `01_eda_tiny.ipynb`.

### 2.1 Dimensões e integridade
| Tabela | Registos |
|---|---:|
| localities | 49 |
| producers | 5 |
| products | 23 |
| collection_centers | 1 |
| vehicles | 2 |
| customers | 8 |
| delivery_locations | 6 |
| orders | 10 |
| order_items | 23 |
| locations | 12 |
| distance_matrix | 132 |

As 10 PK simples e a chave composta da matriz são únicas e não nulas. As 12 relações FK verificadas e os mapeamentos de `locations.ref_id` são válidos. Há 132 = 12 × 11 pares dirigidos, excluindo a diagonal; o modelo atribui distância zero à diagonal.

### 2.2 Procura e capacidade
São **23 artigos de 10 encomendas**, com **248,96 kg** no total. Veículo 1: **1444,70 kg**; veículo 2: **1009,80 kg**; centro: **3232,90 kg**. Ambos os veículos têm `refrigerated=0`.

A procura representa **17,23%** da capacidade do veículo 1 ou **24,65%** do veículo 2. Capacidade em massa suficiente não demonstra uma rota única viável. O máximo por data de abertura da recolha é **43,58 kg em 19/10/2026**.

### 2.3 Janelas temporais
`ready_time = max(order_datetime, availability_start)`. Encomendar após a abertura é permitido, desde que a prontidão não ultrapasse o fecho. **12 artigos** têm prontidão após o fecho: **1, 3, 4, 9, 13, 14, 15, 16, 18, 20, 21 e 23**.

Os outros **11 artigos** são candidatos à recolha, sem garantia de viabilidade da rota. As janelas de entrega duram **1 a 4 horas** e podem ocorrer no dia seguinte à disponibilidade. Os artigos 7, 10 e 22 partilham a data de abertura, mas isso não garante uma rota conjunta viável.

### 2.4 Geografia e matriz
O centro fica em **Terras de Bouro**. Existem cinco produtores e seis destinos; as distâncias do centro aos destinos variam entre **9,14 km e 50,94 km**. O modelo usa as distâncias e durações fornecidas no CSV; as durações são compatíveis, por arredondamento, com cerca de **45 km/h**.

### 2.5 Consequências para o planeamento
Preservar os artigos originais e sinalizar os inviáveis. Validar cada rota com deslocações, espera, capacidade, precedência e regresso ao depósito. Integridade relacional e capacidade agregada não demonstram viabilidade temporal.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

tables = {p.stem: pd.read_csv(p) for p in Path('data').glob('*.csv')}
expected = dict(localities=49, producers=5, products=23, collection_centers=1,
                vehicles=2, customers=8, delivery_locations=6, orders=10,
                order_items=23, locations=12, distance_matrix=132)
assert {name: len(frame) for name, frame in tables.items()} == expected
items = tables['order_items'].merge(tables['orders'], on='order_id').merge(tables['products'], on='product_id')
weight = (items.quantity * items.unit_weight_kg).sum()
ready = pd.concat([pd.to_datetime(items.order_datetime), pd.to_datetime(items.availability_start)], axis=1).max(axis=1)
anomalies = sorted(items.loc[ready > pd.to_datetime(items.availability_end), 'order_item_id'])
assert np.isclose(weight, 248.96)
assert anomalies == [1, 3, 4, 9, 13, 14, 15, 16, 18, 20, 21, 23]
assert tables['vehicles'].refrigerated.eq(0).all()
print(f'CSV confirmados: {weight:.2f} kg, {len(anomalies)} artigos com prontidão após o fecho.')


CSV confirmados: 248.96 kg, 12 artigos com prontidão após o fecho.


## 3. Modeling & Feasibility Tests

`model.py` define `Request`, `Vehicle`, `RouteStep`, `State` e `Problem`. O estado acompanha localização, hora, carga, artigos recolhidos/entregues e custos acumulados.

`actions` respeita capacidade, precedência e início efetivo do serviço após espera. `return_to_base` exige carga vazia e ligação viável. `is_goal` exige os artigos alvo entregues e regresso explícito ao depósito do veículo.

`route_feasibility_checker` exige partida inicial e regresso final ao depósito. Rejeita ações de depósito intermédias, duplicados, entregas sem recolha, carga por entregar, excesso de capacidade, janelas ultrapassadas após espera e ligações inválidas. Os limites temporais são inclusivos; o serviço tem duração zero.

`cost_function` aplica a mesma validação e calcula distância, viagem, espera e duração total. Uma rota inválida devolve `feasible=False` e custo infinito.

As duas rotas manuais válidas mantêm **164,51 km / 241,70 min** e **263,23 km / 351,00 min**. As demonstrações abaixo apresentam dois casos válidos e três inválidos: precedência, janela temporal e capacidade.


In [2]:
import runpy

runpy.run_path('test_handwritten_routes.py', run_name='__main__');


Pedidos válidos: 11

--- Teste Rota Válida 1 (Item 5) ---
Viável: True | Rota perfeitamente viável! Cumpre todas as restrições.
Distância: 164.51 km | Tempo: 241.7 min (Viagem: 219.4 min, Espera: 22.3 min)
  pickup (Item 5) @ Loc 4: Chegada 2026-10-10 13:26:42 -> Saída 2026-10-10 13:26:42 | Espera: 0.0m | Carga: 1.4kg
  delivery (Item 5) @ Loc 10: Chegada 2026-10-10 14:37:42 -> Saída 2026-10-10 15:00:00 | Espera: 22.3m | Carga: 0.0kg
  return (Item 6) @ Loc 6: Chegada 2026-10-10 16:01:42 -> Saída 2026-10-10 16:01:42 | Espera: 0.0m | Carga: 0.0kg

--- Teste Rota Válida 2 (Itens 10 e 7) ---
Viável: True | Rota perfeitamente viável! Cumpre todas as restrições.
Distância: 263.23 km | Tempo: 351.0 min

--- Teste Rota Inválida 1 (Precedência) ---
Viável: False | Violação de Precedência no passo 1: Tentativa de entrega do Item 5 sem ter sido previamente recolhido!

--- Teste Rota Inválida 2 (Janela Temporal) ---
Viável: False | Violação de Janela Temporal de Recolha no passo 1 (Item 5): Iníci

## 4. Problem Solving (Algoritmos de IA)

*(A desenvolver na Entrega Final — Fase 2: Greedy Best-First Search, A* Search com 2 heurísticas admissíveis, Hill-Climbing e CSP Backtracking)*


## 5. Result Analysis & Conclusion

*(A desenvolver na Entrega Final — Fase 2: Tabela comparativa de métricas, análise crítica, limitações e conclusões)*
